# Econ 390: Lecture 20 - More Merging
Today we will be spending some more time merging datasets. Relevant pages are [8.2 in McKinney](https://wesmckinney.com/book/data-wrangling#prep_merge_join), [Merge in Turrell](https://aeturrell.github.io/coding-for-economists/data-joining-data.html#merge), and [this page](https://sscc.wisc.edu/sscc/pubs/dwp/Combining_Data_Sets.html#adding-variables) from the SSCC.

There are three general types of merging you will need to do. In order of how often *I* use them:
1. Creating Panel Data (Same Dataset Across Time)
2. Merging Unrelated Data On Time (Different Datasets Across Time)
3. Merging the Same Data over Multiple Sources (Same Data Across Sources)

We did a little bit of #1 at the end of last lecture. #3 is generally pretty rare and happens when you have high quality restricted data. #2 is very common when you want to control for macro variables that are common among everyone but change over time.

In [1]:
import numpy as np
import pandas as pd

econ390path = "https://m-mcmain.github.io/files/Econ390SP26/"

## Wide and Long Data

![Image showing the difference between wide and long data](https://www.statology.org/wp-content/uploads/2021/12/wideLong1-1-768x543.png "Wide vs. Long Data")

In [2]:
# Create the data
final_four = pd.DataFrame({"Team": ["Michigan", "Arizona", "Illinois", "UConn"],
                           "Points": [91, 73, 62, 71],
                           "Assists": [22, 5, 3, 14],
                           "Rebounds": [40, 44, 44, 37]})
final_four

,Team,Points,Assists,Rebounds
0,Michigan,91,22,40
1,Arizona,73,5,44
2,Illinois,62,3,44
3,UConn,71,14,37


In [7]:
# "Melt" to go from Wide to Long
final_four_melted = pd.melt(final_four, id_vars = "Team")
final_four_melted

,Team,variable,value
0,Michigan,Points,91
1,Arizona,Points,73
2,Illinois,Points,62
3,UConn,Points,71
4,Michigan,Assists,22
5,Arizona,Assists,5
6,Illinois,Assists,3
7,UConn,Assists,14
8,Michigan,Rebounds,40
9,Arizona,Rebounds,44


In [8]:
# Pivot to go back to Wide
final_four_reshape = final_four_melted.pivot(index = "Team", columns = "variable", values = "value")
final_four_reshape

variable,Assists,Points,Rebounds
Team,,,
Arizona,5,73,44
Illinois,3,62,44
Michigan,22,91,40
UConn,14,71,37


In [9]:
# Reset Index
final_four_reshape.reset_index()

variable,Team,Assists,Points,Rebounds
0,Arizona,5,73,44
1,Illinois,3,62,44
2,Michigan,22,91,40
3,UConn,14,71,37


In [10]:
# Select the vars to melt
pd.melt(final_four, id_vars="Team", value_vars = ["Points", "Assists"])

,Team,variable,value
0,Michigan,Points,91
1,Arizona,Points,73
2,Illinois,Points,62
3,UConn,Points,71
4,Michigan,Assists,22
5,Arizona,Assists,5
6,Illinois,Assists,3
7,UConn,Assists,14


## Back to Merging

In [11]:
# Read in data
ffr_cleaned = pd.read_csv(econ390path + "ffr_cleaned.csv", index_col = 0)
rd_BEA = pd.read_csv(econ390path + "rd_BEA.csv", index_col = 0)

In [19]:
# Check out the data
print(ffr_cleaned.tail())
print(rd_BEA.tail())

      FEDFUNDS
Year          
2020      0.38
2021      0.08
2022      1.68
2023      5.02
2024      5.14
        State     RD
Date                
2019  Wyoming  150.7
2020  Wyoming  179.9
2021  Wyoming  182.2
2022  Wyoming  175.8
2023  Wyoming  160.4


In [20]:
# How should we merge them? Concat? Do resulting rows make sense?
rd_merged = pd.merge(ffr_cleaned, rd_BEA, how = "inner", left_index = True, right_index = True)
print(rd_merged.head())
print(rd_merged.tail())

      FEDFUNDS       State       RD
Year                               
2012      0.14     Alabama   2176.2
2012      0.14      Alaska    186.1
2012      0.14     Arizona   4265.9
2012      0.14    Arkansas    439.1
2012      0.14  California  66602.9
      FEDFUNDS          State       RD
Year                                  
2023      5.02       Virginia  10372.0
2023      5.02     Washington  42498.5
2023      5.02  West Virginia    550.4
2023      5.02      Wisconsin   7248.9
2023      5.02        Wyoming    160.4


In [25]:
# How is outer different?
rd_merged_outer = pd.merge(ffr_cleaned, rd_BEA, how = "right", left_index = True, right_index = True, indicator = True) 
print(rd_merged_outer.head())
print(rd_merged_outer.tail())

      FEDFUNDS    State      RD _merge
Year                                  
2012      0.14  Alabama  2176.2   both
2013      0.11  Alabama  2328.3   both
2014      0.09  Alabama  2717.8   both
2015      0.13  Alabama  2480.0   both
2016      0.40  Alabama  3089.6   both
      FEDFUNDS    State     RD _merge
Year                                 
2019      2.16  Wyoming  150.7   both
2020      0.38  Wyoming  179.9   both
2021      0.08  Wyoming  182.2   both
2022      1.68  Wyoming  175.8   both
2023      5.02  Wyoming  160.4   both


## Merging Variable Type

In [27]:
# Set random seed (for reproducibility), ID length, and number of observations
np.random.seed(1)
digits_in_id = 16
n_obs = 100

In [35]:
# Generate string ID of the specified length, convert to float, then back to string
random = np.random.randint(1,10,size=(n_obs, digits_in_id))
id_before = pd.DataFrame(random).astype(str).sum(axis=1)
id_float = id_before.astype(float)
id_after = id_float.astype(str).str[:-2]

In [37]:
# Display results in a DataFrame
id_combined = pd.DataFrame({'id_before': id_before,
                            'id_float':id_float,
                            'id_after':id_after})
id_combined[id_combined['id_before'] != id_combined['id_after']]

,id_before,id_float,id_after
16,9322541199354739,9.322541e+15,9322541199354740
40,9157291833439925,9.157292e+15,9157291833439924
42,9925796753479323,9.925797e+15,9925796753479324
78,9331919189715753,9.331919e+15,9331919189715752


So, if you have data that is read in as Float, you cannot then convert it to a String. It has already done the estimation! So, you'll need to specifically read it in *as* a String by using the `dtype={'var': 'str'}` option in `read_csv`.

Another thing to note when matching with Strings. Make sure that the ID format is consistent across files (e.g. Case Sensitive and Whitespace). If it's not you'll need to do so yourself! Such as `str.lower()` and `str.strip()` respectively.

## Investigating Unmatched Observations

As a reminder, FUSION_annual_1995_EMP is a random sample of 100 Chilean Manufacturing Survey responses from 1995 where each row is one firm that has identifier "NUI" (almost certainly some sort of acronym in Spanish). FUSION_annual_1996_EMP is created similarly, but instead of being a completely random sample, I select the NUI's that show up in both the 1996 Chilean Manufacturing Survey and 1995's, then randomly fill out the rest from 1996 if not all 100 from 1995 are also in 1996. 

Why might not all 100 firms from the random sample in 1995 also be in the 1996 survey responses?

In [40]:
# Read in our data
fusion_1995 = pd.read_stata(econ390path + "FUSION_annual_1995_EMP.dta")
fusion_1996 = pd.read_stata(econ390path + "FUSION_annual_1996_EMP_erroneous.dta")
fusion_1996.head()

,NUI,REGION,CIIU3,TOTHOM,TOTMUJ,REMEMP,REGEMP,REMCOM,REGCOM,REMOBR,REGOBR,TOHSC,TOMSC,EXPORTADOS
0,18784.0,8,1513,88,5,10153,1773,0,0,72243,12614,0,0,1.0
1,13452.0,13,1920,11,0,0,0,0,0,24271,0,0,0,0.0
2,12601.0,13,3311,21,3,36210,1350,2800,210,0,0,0,0,0.0
3,11935.0,13,2519,16,1,0,0,0,0,12596,0,0,0,0.0
4,12335.0,13,3610,39,10,117964,6752,14963,562,74800,3938,0,0,1.0


In [41]:
# Merge
fusion_merged = pd.merge(fusion_1995, fusion_1996, on="NUI", suffixes = ("_1995", "_1996"))
fusion_merged

,NUI,REGION_1995,CIIU3_1995,TOTHOM_1995,TOTMUJ_1995,REMEMP_1995,REGEMP_1995,REMCOM_1995,REGCOM_1995,REMOBR_1995,...,TOTMUJ_1996,REMEMP_1996,REGEMP_1996,REMCOM_1996,REGCOM_1996,REMOBR_1996,REGOBR_1996,TOHSC_1996,TOMSC_1996,EXPORTADOS_1996
0,10082.0,2,3610,14,1,7466,0,0,0,20174,...,1,0,0,0,0,12601,935,0,0,0.0
1,10101.0,2,2720,427,25,2046170,184590,0,0,810717,...,21,3193584,149500,0,0,341581,95786,0,0,1.0
2,10103.0,2,1531,46,6,135873,3426,69814,2912,37037,...,6,179492,4974,44424,744,48379,3820,0,0,0.0
3,10178.0,4,1512,39,35,22958,1886,0,0,40257,...,41,39148,8618,0,0,47848,5518,0,0,1.0
4,10220.0,5,3312,481,15,554636,187853,0,0,858571,...,12,700294,0,0,0,769546,533036,35,4,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,23437.0,13,2520,81,6,88656,1432,3284,0,117469,...,7,116021,6696,0,0,137961,11918,9,0,1.0
87,23449.0,6,2924,145,0,914130,54126,0,0,267928,...,0,1645479,137451,0,0,0,0,0,0,0.0
88,40088.0,8,3693,3,3,13495,271,0,0,0,...,4,17279,347,0,0,0,0,0,0,0.0
89,40172.0,1,2022,31,5,6205,1284,2634,545,64776,...,5,6823,1412,2896,599,71229,532,0,0,0.0


In [16]:
# Check out a specific NUI


In [42]:
# Merge using validate, expecting 1:1
fusion_merged = pd.merge(fusion_1995, fusion_1996, on="NUI", suffixes = ("_1995", "_1996"), how = "outer", validate = "1:1")

MergeError: Merge keys are not unique in right dataset; not a one-to-one merge

In [44]:
# Try Except to save ourselves
try:
    fusion_merged = pd.merge(fusion_1995, fusion_1996, on="NUI", suffixes = ("_1995", "_1996"), how = "outer", validate = "1:1")
except pd.errors.MergeError as e:
    print('WARNING:', e, '-> NUI duplicates dropped\n')
    fusion_merged = pd.merge(fusion_1995, fusion_1996, how="outer", on = "NUI", suffixes = ("_1995", "_1996"))
    fusion_merged = fusion_merged.drop_duplicates(subset="NUI", keep = False)

fusion_merged

,NUI,REGION_1995,CIIU3_1995,TOTHOM_1995,TOTMUJ_1995,REMEMP_1995,REGEMP_1995,REMCOM_1995,REGCOM_1995,REMOBR_1995,...,TOTMUJ_1996,REMEMP_1996,REGEMP_1996,REMCOM_1996,REGCOM_1996,REMOBR_1996,REGOBR_1996,TOHSC_1996,TOMSC_1996,EXPORTADOS_1996
0,10082.0,2.0,3610.0,14.0,1.0,7466.0,0.0,0.0,0.0,20174.0,...,1.0,0.0,0.0,0.0,0.0,12601.0,935.0,0.0,0.0,0.0
1,10101.0,2.0,2720.0,427.0,25.0,2046170.0,184590.0,0.0,0.0,810717.0,...,21.0,3193584.0,149500.0,0.0,0.0,341581.0,95786.0,0.0,0.0,1.0
2,10103.0,2.0,1531.0,46.0,6.0,135873.0,3426.0,69814.0,2912.0,37037.0,...,6.0,179492.0,4974.0,44424.0,744.0,48379.0,3820.0,0.0,0.0,0.0
3,10178.0,4.0,1512.0,39.0,35.0,22958.0,1886.0,0.0,0.0,40257.0,...,41.0,39148.0,8618.0,0.0,0.0,47848.0,5518.0,0.0,0.0,1.0
4,10220.0,5.0,3312.0,481.0,15.0,554636.0,187853.0,0.0,0.0,858571.0,...,12.0,700294.0,0.0,0.0,0.0,769546.0,533036.0,35.0,4.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,23429.0,13.0,2520.0,39.0,0.0,7990.0,790.0,0.0,0.0,25334.0,...,0.0,10931.0,869.0,0.0,0.0,58519.0,4261.0,2.0,0.0,0.0
104,23437.0,13.0,2520.0,81.0,6.0,88656.0,1432.0,3284.0,0.0,117469.0,...,7.0,116021.0,6696.0,0.0,0.0,137961.0,11918.0,9.0,0.0,1.0
105,23449.0,6.0,2924.0,145.0,0.0,914130.0,54126.0,0.0,0.0,267928.0,...,0.0,1645479.0,137451.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
106,40088.0,8.0,3693.0,3.0,3.0,13495.0,271.0,0.0,0.0,0.0,...,4.0,17279.0,347.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## [MovieLens Data](https://wesmckinney.com/book/data-analysis-examples#whetting_movielens)
Are Comedies or Dramas better rated on average? Let's find out!

In [45]:
# Read in the data
movies = pd.read_csv(econ390path + "movies.csv")
ratings = pd.read_csv(econ390path + "ratings.csv")

In [47]:
# Check out the datasets
print(movies.head())
print(ratings.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [51]:
# Merge to create ratings and genre, using validate
ratings_with_genre = pd.merge(movies, ratings, on="movieId", validate="1:m", how="outer", indicator=True)

,movieId,title,genres,userId,rating,timestamp,_merge


In [52]:
# Check out which entries are left or right only
ratings_with_genre[ratings_with_genre["_merge"] == "right_only"]
ratings_with_genre[ratings_with_genre["_merge"] == "left_only"]

,movieId,title,genres,userId,rating,timestamp,_merge
22820,1076,"Innocents, The (1961)",Drama|Horror|Thriller,NaN,NaN,NaN,left_only
49539,2939,Niagara (1953),Drama|Thriller,NaN,NaN,NaN,left_only
53555,3338,For All Mankind (1989),Documentary,NaN,NaN,NaN,left_only
54467,3456,"Color of Paradise, The (Rang-e khoda) (1999)",Drama,NaN,NaN,NaN,left_only
60535,4194,I Know Where I'm Going! (1945),Drama|Romance|War,NaN,NaN,NaN,left_only
68396,5721,"Chosen, The (1981)",Drama,NaN,NaN,NaN,left_only
71896,6668,"Road Home, The (Wo de fu qin mu qin) (1999)",Drama|Romance,NaN,NaN,NaN,left_only
72428,6849,Scrooge (1970),Drama|Fantasy|Musical,NaN,NaN,NaN,left_only
73354,7020,Proof (1991),Comedy|Drama|Romance,NaN,NaN,NaN,left_only
75450,7792,"Parallax View, The (1974)",Thriller,NaN,NaN,NaN,left_only


In [57]:
# Re-run to exclude unmatched observations
ratings_with_genre = pd.merge(movies, ratings, on="movieId", validate="1:m", how="inner")
ratings_with_genre

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1,4.0,964982703
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5,4.0,847434962
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,7,4.5,1106635946
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,15,2.5,1510577970
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,17,4.5,1305696483
...,...,...,...,...,...,...
100831,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy,184,4.0,1537109082
100832,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy,184,3.5,1537109545
100833,193585,Flint (2017),Drama,184,3.5,1537109805
100834,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation,184,3.5,1537110021


In [56]:
# Ratings by Genre - does this make sense?
print("Descriptive Statistics for Comedies:")
print(ratings_with_genre[ratings_with_genre['genres']=="Comedy"]['rating'].describe())
print("Descriptive Statistics for Drama:")
print(ratings_with_genre[ratings_with_genre['genres']=="Drama"]['rating'].describe())

Descriptive Statistics for Comedies:
count    7196.000000
mean        3.197888
std         1.107383
min         0.500000
25%         2.500000
50%         3.000000
75%         4.000000
max         5.000000
Name: rating, dtype: float64
Descriptive Statistics for Drama:
count    6291.000000
mean        3.688841
std         0.920846
min         0.500000
25%         3.000000
50%         4.000000
75%         4.000000
max         5.000000
Name: rating, dtype: float64


In [58]:
# Properly accommodate genres
print("Descriptive Statistics for Comedies:")
print(ratings_with_genre[ratings_with_genre['genres'].str.contains("Comedy")]['rating'].describe())
print("Descriptive Statistics for Drama:")
print(ratings_with_genre[ratings_with_genre['genres'].str.contains("Drama")]['rating'].describe())

Descriptive Statistics for Comedies:
count    39053.000000
mean         3.384721
std          1.066541
min          0.500000
25%          3.000000
50%          3.500000
75%          4.000000
max          5.000000
Name: rating, dtype: float64
Descriptive Statistics for Drama:
count    41928.000000
mean         3.656184
std          0.979133
min          0.500000
25%          3.000000
50%          4.000000
75%          4.500000
max          5.000000
Name: rating, dtype: float64
